# Transformer(4.Masked Multi-Head Attention) 언어 모델 총정리 

## 1. Decoder Block
	◦ 트랜스포머 디코더 블록 단계별 공식
	1) 입력값(token) 
	2) 임베딩 + 위치인코딩 
	3) Masked Multi-Head Attention 
	4) 1차 Residual Connection → 1차 Layer Normalization
	5) Cross Attention(인코더 Key/Value, 디코더 Query 사용하여 Attention 계산) 
	6) 2차 Residual Connection → 2차 Layer Normalization 
	7) Feed Forward Network(FFN)
	8) 3차 Residual Connection → 3차 Layer Normalization

---
### 1) 임베딩 + 위치 인코딩
$ [ Y = \text{Embedding}(tokens) + \text{PositionalEncoding} ] $

### 2) Masked Multi-Head Attention
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

### 3) Residual Connection + Layer Normalization (1차)
$ [ H_1 = \text{LayerNorm}(Y + \text{MaskedMHA}(Y)) ] $

### 4) Cross Attention (인코더 Key/Value, 디코더 Query 사용하여 Attention 계산)
$ [ \text{CrossAttn}(H_1, H_{enc}) = \text{Softmax}\left(\frac{Q_{dec}K_{enc}^\top}{\sqrt{d_k}}\right)V_{enc} ] $

### 5) Residual Connection + Normalization (2차)
$ [ H_2 = \text{LayerNorm}(H_1 + \text{CrossAttn}(H_1, H_{enc})) ] $

### 6) Feed Forward Network (FFN)
$ [ \text{FFN}(H_2) = \max(0, H_2 W_1 + b_1) W_2 + b_2 ] $

### 7) Residual Connection + LayerNorm (3차)
$ [ H_3 = \text{LayerNorm}(H_2 + \text{FFN}(H_2)) ] $

---

## 2. Transformer(4.Masked Multi-Head Attention) 언어 모델의 Masked Multi-Head Attention 계산 과정을 공식과 함께 단계별로 계산
---
### 1) 입력 벡터 + 위치 인코딩
디코더 입력 토큰 "I"를 임베딩 후 위치 인코딩을 더합니다.

$ [ Y = \text{Embedding}("I") + \text{PositionalEncoding} ] $

예: $ 디코더 입력 "I" -> 입베딩 벡터[0.5, 1.2] , 위치인코딩[0.1, -0.1] $

$ [ Y = [0.5, 1.1] + [0.1, -0.1] ] $

$ [ Y = [0.6, 1.1] ] $

---
### 2) Query / Key / Value 생성
	◦ Attention 계산(Query/Key/Value 생성 단계)
$ [ Q = Y W_Q, \quad K = Y W_K, \quad V = Y W_V ] $

2-1) 예시 가중치 행렬(초기에 가중치는 램덤 설정된다) :
$ \text{WQ} = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} , $
$ \text{WK} = \begin{bmatrix}0.5 & 0 \\ 0 & 0.5\end{bmatrix} , $
$ \text{WV} = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} $

2-2) Query / Key / Value 생성(계산) 결과 :

$ Q = [0.6, 1.1] \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} = [0.6, 1.1] $

$ K = [0.6, 1.1] \begin{bmatrix}0.5 & 0 \\ 0 & 0.5\end{bmatrix} = [0.3, 0.55] $ 

$ V = [0.6, 1.1] \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} = [0.6, 1.1] $

---
### 3) 벡터 간 유사도 계산 (내적)
	◦ Attention 계산(유사도 내적 계산 단계)
$ [ QK^\top ] $

Query와 Key의 내적을 통해 유사도를 계산합니다 : 

$ QK^\top = [0.6, 1.1] \begin{bmatrix}0.3 \\ 0.5\end{bmatrix} $

$ QK^\top = 0.6 \cdot 0.3 + 1.1 \cdot 0.55 = 0.785 $

$ QK^\top = 0.785 $

---
### 4) 스케일링
	◦ Attention 계산(Scaled Dot-Product 단계)
$ [ \text{Scaled Score} = \frac{QK^\top}{\sqrt{d_k}} ] $

내적 $ QK^\top $ 값의 차원이 커질수록 커지게 되며, 값이 너무 커지면 Softmax에서 기울기 소실(Gradient Vanishing) 문재가 발생

따라서 $ [ \sigma = \sqrt{d_k} ] $ 로 나누어 값이 안정화되어 Softmax가 잘 작동됨, 
참고) Key 벡터 차원 $ [ d_k = 2 ] $

$ \text{Scaled Score} = \frac{QK^\top}{\sqrt{d_k}} $

$ = \frac{0.785}{\sqrt{2}} $

$ = \frac{0.785}{{1.414}} $

$ \approx 0.555 $

---
### 5) Softmax (마스크 포함)
	◦ 소트프맥스 기본 공식은 : 
$ [ P_i = \frac{e^{z_i}}{\sum_j e^{z_j}} ] $ 

$ [ z_i ] $ 특정 단어(토큰)에 대한 스케일된 Score, 
$ [ z_i = \frac{QK_i^\top}{\sqrt{d_k}} ] $
즉, Query와 해당 단어의 Key 벡터 내적을 스케일링한 값입니다.

$ [ z_j ] $ 모든 단어(토큰)에 대한 스케일된 Score 집합,
$ [ z_j = \frac{QK_j^\top}{\sqrt{d_k}}, \quad j = 1,2,\ldots,n ] $
즉, 현재 시점에서 볼 수 있는 모든 단어의 스코어를 의미합니다.

	◦ Attention 계산(Softmax 단계) 
$ [ \text{Score} = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right) ] $

만약 단어가 2개 있다고 가정하면 (현재 단어 "I", 미래 단어 "am"), 마스크 때문에 "am"은 $ [ -\infty ] $(무한)로 처리된다.

따라서 Softmax 입력값은 :
앞서 스케일링한 값 :
$ [ \frac{QK^\top}{\sqrt{d_k}} = 0.555 ] $

$ [ z = [0.555, -\infty] ] $ 

Softmax 계산: $ 참고) [ e^{0.555} = 1.742 ] , [ e^{-\infty} = 0 ] $

$ P(\text{"I"}) = \frac{e^{0.555}}{e^{0.555} + e^{-\infty}} $

$ = \frac{e^{1.742}}{e^{1.742} + 0} = 1.0 $

$ P(\text{"am"}) = 0 $

	◦ 현재 단어 "I"에 확률 1.0이 집중됨
	◦ 미래 단어 "am"은 마스크 때문에 확률 0
	◦ 따라서 Attention은 현재 단어 "I"의 Value 벡터만 가져오게 됩니다.

---
### 6) Value 가중합
	◦ Attention 계산(Value 가중합 단계)
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

Softmax 확률을 Value 벡터에 곱해 최종 Attention 출력을 얻습니다

$ \text{MaskedMHA}(Y) = P(\text{"I"}) \cdot V + P(\text{"am"}) \cdot V_{am} $

$ = 1.0 \cdot [0.6, 1.1] + 0 \cdot V_{am} $

$ = [0.6, 1.1] $

---